In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math
import time

## Batched causal linear attention

We keep the unnormalized recurrence: `S_t = S_(t-1) + outer(k_t, v_t)` and `o_t = q_t @ S_t`.

Inputs `q` and `k` have shape `[B, T, d_k]`; `v` has shape `[B, T, d_v]`.
`B` is the number of independent sequences and `T` is their length.
The outer loop picks one sequence. It starts a fresh memory matrix `S` of shape `[d_k, d_v]`.
The inner loop advances through that sequence using `torch.outer` and ordinary matrix multiplication.
We stack the token outputs, then stack the sequences to return `[B, T, d_v]`.

`torch.outer` takes two one-dimensional vectors, so we process sequences separately.
This is a readable learning implementation; the extra Python loop can be slower than batched operations.
For now we supply Q/K/V directly. Learned projections from an input `x` come later.


In [6]:
class CausalLinearAttention(nn.Module):
    def forward(self, q, k, v):
        # q, k: [B, T, d_k]; v: [B, T, d_v]
        B, T, d_k = q.shape
        d_v = v.shape[-1]
        batch_outputs = []

        for b in range(B):
            # each sequence starts with its own memory
            S = q.new_zeros(d_k, d_v)
            outputs = []

            for t in range(T):
                # [d_k] outer [d_v] -> [d_k, d_v]
                S = S + torch.outer(k[b, t], v[b, t])
                # [d_k] @ [d_k, d_v] -> [d_v]
                output_t = q[b, t] @ S
                outputs.append(output_t)

            batch_outputs.append(torch.stack(outputs))  # [T, d_v]

        return torch.stack(batch_outputs)  # [B, T, d_v]


In [7]:
torch.manual_seed(0)

B, T, d_k, d_v = 2, 4, 3, 2
# float64 makes the numerical comparison stricter
# requires_grad=True lets us check derivatives with respect to all inputs.
q = torch.randn(B, T, d_k, dtype=torch.float64, requires_grad=True)
k = torch.randn(B, T, d_k, dtype=torch.float64, requires_grad=True)
v = torch.randn(B, T, d_v, dtype=torch.float64, requires_grad=True)

attention = CausalLinearAttention()
o = attention(q, k, v)
print("Output shape:", o.shape)  # [2, 4, 2]


Output shape: torch.Size([2, 4, 2])
